# Alle Finetune-Schritte vs. Baseline — Gesamtauswertung

Vergleich aller bisher trainierten Modelle auf dem Kiel-Testset (kronenweise, IoU 0.5):

| Modell | Trainingsdaten | Kanäle |
|---|---|---|
| `baseline` | un-finetunt (`freudenberg2022`, ~20cm Sommer) | 5 (RGBI+NDVI) |
| `step1_spring75` | 100% Frühjahr **7.5cm** | 5 |
| `step2_spring20` | 100% Frühjahr **20cm** | 5 |
| `step3_mix20` | **50/50** Sommer+Frühjahr **20cm** | 5 |

⚠️ **Postprocessing (PP) ist pro Modell×Auflösung getunt** (siehe `pp_sweep.py`). Die
`eval_test.csv` unten wurden zu unterschiedlichen Zeitpunkten erzeugt: Baseline/step1/step2
mit Default-PP **10/1**, step3 mit **10/3**. Die Live-Matrix ist daher *nicht* durchgängig
apples-to-apples — die belastbaren, getunten Heim-Auflösungs-Werte stehen in Abschnitt 2.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

R = Path('/home/leafline/leafline/3_Model/runs')
COLORS = {'baseline':'#eda100','step1_spring75':'#2a78d6','step2_spring20':'#008300','step3_mix20':'#8a4fbe'}
RES_ORDER = ['7.5cm','20cm','20cm-spring']

def load(p, run=None):
    p=Path(p)
    if not p.exists(): print(f'FEHLT: {p}'); return None
    df=pd.read_csv(p)
    if run: df['run']=run
    print(f'geladen: {p.relative_to(R)} ({len(df)} Zeilen)')
    return df

evals = {
    'baseline':       load(R/'baseline_eval.csv','baseline'),
    'step1_spring75': load(R/'step1_spring75/eval_test.csv','step1_spring75'),
    'step2_spring20': load(R/'step2_spring20/eval_test.csv','step2_spring20'),
    'step3_mix20':    load(R/'step3_mix20/eval_test.csv','step3_mix20'),
}
train_logs = {n: load(R/f'{n}/train_log.csv') for n in
              ['step1_spring75','step2_spring20','step3_mix20']}

## 1. Trainingsverlauf — pixelweise Val-F1

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
for run, tl in train_logs.items():
    if tl is None: continue
    ax.plot(tl['epoch'], tl['val_f1'], color=COLORS[run], lw=2, label=run)
    bi=tl['val_f1'].idxmax(); ax.scatter([tl.loc[bi,'epoch']],[tl.loc[bi,'val_f1']],color=COLORS[run],zorder=5)
    print(f"{run}: bestes val_F1 {tl['val_f1'].max():.4f} @ ep{int(tl.loc[bi,'epoch'])}")
ax.set_xlabel('Epoche'); ax.set_ylabel('Val-F1 (pixelweise)')
ax.set_title('Trainingsverlauf'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 2. Kronenweise Test-F1

### 2a. Belastbare Werte: bestes Modell je Domäne, mit *getuntem* PP

| Domäne | bestes Modell | getuntes PP | F1 |
|---|---|---|---|
| **7.5cm** (Frühjahr) | step1 | 30/2 | **0.150** |
| **20cm** (Sommer) | step3 | 10/3 | **0.317** (Baseline 0.340) |
| **20cm-spring** (Frühjahr) | step3 | 10/3 | **0.144** (step2 0.113) |

Kernaussagen:
- Getrennte Modelle je **Auflösung** (7.5 vs. 20), aber **ein** Modell je Auflösung über
  beide **Saisons**: step3 (50/50) hält Sommer *und* Frühjahr und ist bei 20cm-spring sogar
  besser als das reine Frühjahrsmodell step2 → Schedule-Schritt 4 (separates Sommer-Modell)
  ist nicht nötig.
- Baseline ist bei 20cm-Sommer knapp vorn (0.340), aber step3 ist ein *generelles* 20cm-Modell,
  das auch Frühjahr kann — für Kiels Frühjahrs-Workflow ist step3 das beste 20cm-Modell.

### 2b. Live-Matrix aus den eval_test.csv (⚠️ gemischtes PP, siehe Kopf)

In [ ]:
def micro(df):
    out={}
    for res,g in df.groupby('aufloesung'):
        tp,fp,fn=g.tp.sum(),g.fp.sum(),g.fn.sum()
        p=tp/(tp+fp) if tp+fp else 0.0; r=tp/(tp+fn) if tp+fn else 0.0
        out[res]=2*p*r/(p+r) if p+r else 0.0
    return out

tbl = pd.DataFrame({run: micro(df) for run,df in evals.items() if df is not None}).reindex(RES_ORDER)
print('Mikro-F1 je Modell × Auflösung (eval_test.csv):')
print(tbl.round(4).to_string())

## 3. Vergleichsdiagramm — F1 je Auflösung

In [ ]:
runs=[r for r in ['baseline','step1_spring75','step2_spring20','step3_mix20'] if r in tbl.columns]
x=np.arange(len(RES_ORDER)); w=0.2
fig, ax = plt.subplots(figsize=(11,5.5))
for k,run in enumerate(runs):
    vals=[tbl.loc[res,run] if (res in tbl.index and run in tbl.columns and not np.isnan(tbl.loc[res,run])) else 0 for res in RES_ORDER]
    off=(k-(len(runs)-1)/2)*w
    ax.bar(x+off, vals, width=w, label=run, color=COLORS[run])
    for xi,v in zip(x+off,vals):
        if v>0: ax.text(xi, v+0.005, f'{v:.3f}', ha='center', va='bottom', fontsize=7)
ax.set_xticks(x); ax.set_xticklabels(RES_ORDER)
ax.set_ylabel('Mikro-F1 (kronenweise, IoU 0.5)')
ax.set_ylim(0, max(0.4, np.nanmax(tbl.values)*1.25))
ax.set_title('Baseline vs. Finetune-Schritte je Auflösung  (PP: base/step1/step2=10/1, step3=10/3)')
ax.legend(); ax.grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.show()

## 4. Befunde & Schedule-Stand

**1. Finetuning gewinnt auf allen Frühjahrs-Domänen.** 7.5cm 0.000→0.150 (step1),
20cm-spring 0.044→0.144 (step3).

**2. Auflösung braucht getrennte Modelle.** Ein 7.5cm-Modell bricht bei 20cm ein und umgekehrt
(die Diagonale in den früheren Vergleichen). → step1 für 7.5cm, ein 20cm-Modell für 20cm.

**3. Saison braucht *kein* getrenntes Modell.** step3 (50/50) hält beide Jahreszeiten bei 20cm
(Sommer 0.317 ≈ Baseline 0.340; Frühjahr 0.144 > step2 0.113) — die zusätzlichen Sommerdaten
verbessern sogar das Frühjahr (Recall steigt bei beiden). → **Schedule-Schritt 4 entfällt.**

**4. PP und LR sind nicht der Hebel.** pp_sweep + CV-LR-Suche zeigten: PP hebt nur die Precision
(Über-Segmentierung), der Recall bleibt die Decke; LR-Tuning brachte auf dem Test nichts
(step2-cvlr: val 0.696→0.802, aber Test 0.113→0.101). → der verbleibende Hebel ist Information/
Kapazität: **nDOM (Höhe)**.

**Schedule-Stand:** Schritt 1 ✅, 2 ✅, 3 ✅, 4 ⛔ (nicht nötig). Offen: **4b/5 — Höhenkanal
(nDOM)**; Config `finetune_step1_ndom_spring75.yaml` liegt bereit (step1 mit in_channels=6).
